<a href="https://colab.research.google.com/github/Phamminhtriet-dotcom/Appmophonggiaohang.py/blob/main/Group1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
"""
╔══════════════════════════════════════════════════════════════════╗
║   🛵 HỆ THỐNG ĐÁNH GIÁ TÀI XẾ GIAO ĐỒ ĂN NHANH               ║
║      Food Delivery Driver Evaluation System                      ║
║      Sử dụng Logic Mờ (Fuzzy Logic)                             ║
║                                                                  ║
║   📚 Giáo trình: Trí Tuệ Nhân Tạo Trong Quản Trị Công Nghệ     ║
║   🖥️  Tương thích: Google Colab                                  ║
╚══════════════════════════════════════════════════════════════════╝

HƯỚNG DẪN SỬ DỤNG TRÊN GOOGLE COLAB:
  1. Tải file này lên Colab (hoặc copy toàn bộ nội dung vào 1 cell)
  2. Chạy: exec(open('food_delivery_evaluation.py').read())
     HOẶC copy toàn bộ vào cell và nhấn Shift+Enter
  3. Chờ cài đặt thư viện (~30 giây lần đầu)
  4. Giao diện sẽ hiện ra ngay bên dưới
"""

# ================================================================
# BƯỚC 1: CÀI ĐẶT & IMPORT THƯ VIỆN
# ================================================================
import subprocess, sys

print("⏳ Đang cài đặt thư viện cần thiết...")
for pkg in ['folium', 'geopy', 'scikit-fuzzy']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print("✅ Cài đặt hoàn tất!\n")

import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import folium
from geopy.distance import geodesic
import warnings
warnings.filterwarnings('ignore')

# ================================================================
# BƯỚC 2: CẤU HÌNH HỆ THỐNG
# ================================================================

# --- Địa điểm mặc định tại TP. Hồ Chí Minh ---
LOCATIONS = {
    "🏙️ Quận 1 – Bến Thành":           (10.7721, 106.6982),
    "🏙️ Quận 1 – Phạm Ngũ Lão":         (10.7692, 106.6947),
    "🏘️ Quận 3 – Võ Văn Tần":           (10.7757, 106.6889),
    "🏘️ Quận 5 – Chợ Lớn":             (10.7543, 106.6602),
    "🏢 Quận 7 – Phú Mỹ Hưng":          (10.7296, 106.7218),
    "🏘️ Quận 10 – Nguyễn Tri Phương":   (10.7742, 106.6768),
    "🌿 Bình Thạnh – Văn Thánh":        (10.8026, 106.7136),
    "✈️ Tân Bình – Sân Bay TSN":        (10.8124, 106.6556),
    "🏘️ Gò Vấp – Quang Trung":          (10.8273, 106.6733),
    "🎓 Thủ Đức – Làng Đại Học":        (10.8700, 106.8017),
    "🌊 Nhà Bè – Phú Xuân":             (10.6941, 106.7399),
    "🏡 Hóc Môn – Trung Tâm":           (10.8899, 106.5956),
    "🎪 Bình Dương – Aeon Mall":         (10.9779, 106.7112),
    "🌾 Củ Chi – Trung Tâm":            (10.9718, 106.4731),
}

# --- Cấu hình giá ---
GIA_CO_BAN   = 15_000   # VND (phí mở cửa)
GIA_THEO_KM  = 5_000    # VND/km
HE_SO_THOI_TIET = {
    'clear':  1.0,
    'rainy':  1.2,
    'stormy': 1.5,
}

# ================================================================
# BƯỚC 3: XÂY DỰNG HỆ THỐNG LOGIC MỜ
# ================================================================

def _tao_bien_dau_vao():
    """Tạo các biến đầu vào mờ (Antecedents)"""
    traffic  = ctrl.Antecedent(np.arange(0, 11, 0.1), 'traffic')
    distance = ctrl.Antecedent(np.arange(0, 21, 0.1), 'distance')
    weather  = ctrl.Antecedent(np.arange(0, 11, 0.1), 'weather')
    prep     = ctrl.Antecedent(np.arange(0, 31, 0.1), 'prep')
    fatigue  = ctrl.Antecedent(np.arange(0, 11, 0.1), 'fatigue')

    # Giao thông (0=thông thoáng, 10=tắc nghẽn nặng)
    traffic['low']    = fuzz.trapmf(traffic.universe,  [0, 0, 2, 4])
    traffic['medium'] = fuzz.trimf(traffic.universe,   [2, 5, 8])
    traffic['high']   = fuzz.trapmf(traffic.universe,  [6, 8, 10, 10])

    # Khoảng cách (km)
    distance['short']  = fuzz.trapmf(distance.universe, [0, 0, 2, 4])
    distance['medium'] = fuzz.trimf(distance.universe,  [2, 5.5, 9])
    distance['long']   = fuzz.trapmf(distance.universe, [7, 10, 20, 20])

    # Thời tiết (0=quang đãng, 10=giông bão)
    weather['clear']  = fuzz.trapmf(weather.universe, [0, 0, 2, 4])
    weather['rainy']  = fuzz.trimf(weather.universe,  [3, 5, 7])
    weather['stormy'] = fuzz.trapmf(weather.universe, [6, 8, 10, 10])

    # Thời gian chuẩn bị đơn hàng (phút)
    prep['fast']   = fuzz.trapmf(prep.universe, [0, 0, 3, 6])
    prep['medium'] = fuzz.trimf(prep.universe,  [4, 10, 16])
    prep['slow']   = fuzz.trapmf(prep.universe, [14, 18, 30, 30])

    # Mức độ mệt mỏi tài xế (0=khỏe, 10=kiệt sức)
    fatigue['low']    = fuzz.trapmf(fatigue.universe, [0, 0, 2, 4])
    fatigue['medium'] = fuzz.trimf(fatigue.universe,  [3, 5, 7])
    fatigue['high']   = fuzz.trapmf(fatigue.universe, [6, 8, 10, 10])

    return traffic, distance, weather, prep, fatigue


def xay_dung_he_thong_mo():
    """
    Xây dựng 3 hệ thống điều khiển mờ riêng biệt:
      1. Thời gian giao hàng ước tính
      2. Tiền thưởng khuyến khích
      3. Xếp hạng hiệu suất tài xế
    """
    traffic, distance, weather, prep, fatigue = _tao_bien_dau_vao()

    # ── HỆ THỐNG 1: Thời gian giao hàng (phút) ──────────────────
    del_time = ctrl.Consequent(np.arange(0, 61, 0.1), 'del_time')
    del_time['short']  = fuzz.trapmf(del_time.universe, [0, 0, 7, 12])
    del_time['medium'] = fuzz.trimf(del_time.universe,  [9, 20, 30])
    del_time['long']   = fuzz.trapmf(del_time.universe, [26, 33, 60, 60])

    rules_time = [
        # Luật dựa trên giao thông & khoảng cách
        ctrl.Rule(traffic['low']    & distance['short'],  del_time['short']),
        ctrl.Rule(traffic['medium'] & distance['medium'], del_time['medium']),
        ctrl.Rule(traffic['high']   & distance['long'],   del_time['long']),
        ctrl.Rule(traffic['low']    & distance['medium'], del_time['medium']),
        ctrl.Rule(traffic['low']    & distance['long'],   del_time['medium']),
        ctrl.Rule(traffic['medium'] & distance['short'],  del_time['short']),
        ctrl.Rule(traffic['medium'] & distance['long'],   del_time['long']),
        ctrl.Rule(traffic['high']   & distance['short'],  del_time['medium']),
        ctrl.Rule(traffic['high']   & distance['medium'], del_time['long']),

        # Luật dựa trên thời gian chuẩn bị & giao thông
        ctrl.Rule(prep['fast']   & traffic['low'],    del_time['short']),
        ctrl.Rule(prep['medium'] & traffic['medium'], del_time['medium']),
        ctrl.Rule(prep['slow']   & traffic['high'],   del_time['long']),
        ctrl.Rule(prep['fast']   & traffic['high'],   del_time['medium']),
        ctrl.Rule(prep['slow']   & traffic['low'],    del_time['medium']),

        # Luật kết hợp nhiều yếu tố
        ctrl.Rule(distance['long']  & weather['stormy'] & traffic['high'],  del_time['long']),
        ctrl.Rule(distance['short'] & weather['clear']  & traffic['low'],   del_time['short']),
        ctrl.Rule(fatigue['high']   & traffic['high'],                       del_time['long']),
        # Luật image 1: khoảng cách ngắn + quang đãng + giao thông thấp
        ctrl.Rule(distance['short'] & weather['clear']  & traffic['low'],   del_time['short']),
        # Luật image 1: tài xế mệt + giao thông đông đúc
        ctrl.Rule(fatigue['high']   & traffic['high'],                       del_time['long']),
    ]

    sys_time = ctrl.ControlSystem(rules_time)
    sim_time = ctrl.ControlSystemSimulation(sys_time)

    # ── HỆ THỐNG 2: Tiền thưởng khuyến khích (%) ────────────────
    bonus = ctrl.Consequent(np.arange(0, 101, 0.1), 'bonus')
    bonus['low']    = fuzz.trapmf(bonus.universe, [0,  0,  10, 20])
    bonus['medium'] = fuzz.trimf(bonus.universe,  [15, 30, 50])
    bonus['high']   = fuzz.trapmf(bonus.universe, [40, 60, 100, 100])

    rules_bonus = [
        # Luật dựa trên thời tiết
        ctrl.Rule(weather['clear'],  bonus['low']),
        ctrl.Rule(weather['rainy'],  bonus['medium']),
        ctrl.Rule(weather['stormy'], bonus['high']),

        # Luật kết hợp
        ctrl.Rule(distance['long']  & weather['stormy'] & traffic['high'],  bonus['high']),
        ctrl.Rule(distance['short'] & weather['clear']  & traffic['low'],   bonus['low']),
        # image 1: khoảng cách ngắn + quang đãng + giao thông thấp → thưởng thấp
        ctrl.Rule(distance['short'] & weather['clear'],  bonus['low']),
    ]

    sys_bonus = ctrl.ControlSystem(rules_bonus)
    sim_bonus = ctrl.ControlSystemSimulation(sys_bonus)

    # ── HỆ THỐNG 3: Xếp hạng hiệu suất (1–5 sao) ────────────────
    rating = ctrl.Consequent(np.arange(1, 5.01, 0.01), 'rating')
    rating['poor']      = fuzz.trapmf(rating.universe, [1, 1, 1.5, 2.5])
    rating['average']   = fuzz.trimf(rating.universe,  [2, 3,  4])
    rating['excellent'] = fuzz.trapmf(rating.universe, [3.5, 4.5, 5, 5])

    rules_rating = [
        # Luật dựa trên mức độ mệt mỏi
        ctrl.Rule(fatigue['low'],    rating['excellent']),
        ctrl.Rule(fatigue['medium'], rating['average']),
        ctrl.Rule(fatigue['high'],   rating['poor']),

        # Luật kết hợp
        ctrl.Rule(fatigue['high'] & traffic['high'], rating['poor']),
        ctrl.Rule(fatigue['low']  & traffic['low'],  rating['excellent']),
    ]

    sys_rating = ctrl.ControlSystem(rules_rating)
    sim_rating = ctrl.ControlSystemSimulation(sys_rating)

    return sim_time, sim_bonus, sim_rating


def chay_he_thong_mo(sim_time, sim_bonus, sim_rating,
                     traffic_val, distance_val, weather_val,
                     prep_val, fatigue_val):
    """Thực thi suy diễn mờ, trả về (del_time, bonus_pct, rating)"""
    dist_clip = float(min(distance_val, 20.0))
    prep_clip = float(min(prep_val, 30.0))
    t = float(traffic_val)
    w = float(weather_val)
    f = float(fatigue_val)

    # -- Thời gian giao hàng --
    try:
        sim_time.input['traffic']  = t
        sim_time.input['distance'] = dist_clip
        sim_time.input['weather']  = w
        sim_time.input['prep']     = prep_clip
        sim_time.input['fatigue']  = f
        sim_time.compute()
        del_time = sim_time.output['del_time']
    except Exception:
        # Fallback tuyến tính đơn giản
        del_time = 5 + dist_clip * 2 + t * 1.5 + prep_clip * 0.3

    # -- Tiền thưởng --
    try:
        sim_bonus.input['traffic']  = t
        sim_bonus.input['distance'] = dist_clip
        sim_bonus.input['weather']  = w
        sim_bonus.input['prep']     = prep_clip
        sim_bonus.input['fatigue']  = f
        sim_bonus.compute()
        bonus_pct = sim_bonus.output['bonus']
    except Exception:
        bonus_pct = w * 4.0  # fallback

    # -- Xếp hạng --
    try:
        sim_rating.input['traffic']  = t
        sim_rating.input['distance'] = dist_clip
        sim_rating.input['weather']  = w
        sim_rating.input['prep']     = prep_clip
        sim_rating.input['fatigue']  = f
        sim_rating.compute()
        rating_val = sim_rating.output['rating']
    except Exception:
        rating_val = max(1.0, 5.0 - f * 0.4)  # fallback

    return del_time, bonus_pct, rating_val


# ================================================================
# BƯỚC 4: HÀM TIỆN ÍCH
# ================================================================

def tinh_gia(distance_km: float, weather_val: float) -> tuple:
    if weather_val <= 3:
        weather_key, weather_lbl = 'clear', 'Quang đãng ☀️'
    elif weather_val <= 6:
        weather_key, weather_lbl = 'rainy', 'Mưa 🌧️'
    else:
        weather_key, weather_lbl = 'stormy', 'Giông bão ⛈️'

    gia_km    = GIA_CO_BAN + distance_km * GIA_THEO_KM
    surge     = HE_SO_THOI_TIET[weather_key]
    gia_final = gia_km * surge
    return gia_final, weather_lbl, surge


def hien_thi_sao(rating: float) -> str:
    full  = int(rating)
    half  = 1 if (rating - full) >= 0.3 else 0
    empty = 5 - full - half
    return '⭐' * full + ('✨' if half else '') + '☆' * empty


def tao_ban_do(pickup_coords, delivery_coords, pickup_name, delivery_name) -> folium.Map:
    center = ((pickup_coords[0] + delivery_coords[0]) / 2,
              (pickup_coords[1] + delivery_coords[1]) / 2)
    m = folium.Map(location=list(center), zoom_start=13, tiles='OpenStreetMap')

    folium.Marker(
        list(pickup_coords),
        popup=f"<b>🏪 Điểm lấy hàng</b><br><small>{pickup_name}</small>",
        tooltip="🏪 Lấy hàng",
        icon=folium.Icon(color='green', icon='shopping-cart', prefix='fa'),
    ).add_to(m)

    folium.Marker(
        list(delivery_coords),
        popup=f"<b>🏠 Điểm giao hàng</b><br><small>{delivery_name}</small>",
        tooltip="🏠 Giao hàng",
        icon=folium.Icon(color='red', icon='home', prefix='fa'),
    ).add_to(m)

    folium.PolyLine(
        [list(pickup_coords), list(delivery_coords)],
        color='#3498DB', weight=4, opacity=0.85, dash_array='10',
        tooltip=f"Khoảng cách: {geodesic(pickup_coords, delivery_coords).km:.2f} km",
    ).add_to(m)

    m.fit_bounds([list(pickup_coords), list(delivery_coords)],
                 max_zoom=15, padding=[40, 40])
    return m


# ================================================================
# BƯỚC 5: GIAO DIỆN NGƯỜI DÙNG (ipywidgets)
# ================================================================

def tao_ung_dung():
    """Tạo toàn bộ giao diện ứng dụng"""

    # Khởi tạo hệ thống mờ
    print("⚙️  Đang khởi tạo hệ thống Logic Mờ...")
    sim_time, sim_bonus, sim_rating = xay_dung_he_thong_mo()
    print("✅  Hệ thống sẵn sàng!\n")

    loc_names = list(LOCATIONS.keys())

    # ── TIÊU ĐỀ ────────────────────────────────────────────────
    display(HTML("""
    <div style="
        background: linear-gradient(135deg, #FF6B35 0%, #F7931E 50%, #FFD700 100%);
        padding: 22px 30px; border-radius: 18px; margin-bottom: 18px; text-align: center;
        box-shadow: 0 6px 20px rgba(255,107,53,0.35);">
      <h1 style="color:white; margin:0; font-size:28px; letter-spacing:1px;">
        🛵 Hệ Thống Đánh Giá Tài Xế Giao Đồ Ăn
      </h1>
      <p style="color:#FFF3E0; margin:6px 0 0; font-size:15px;">
        Food Delivery Driver Evaluation System &nbsp;|&nbsp; Fuzzy Logic (Logic Mờ)
      </p>
    </div>
    """))

    # ═══════════════════════════════════════════════════════════
    # PHẦN 1 – ĐỊA ĐIỂM
    # ═══════════════════════════════════════════════════════════
    display(HTML("""
    <h3 style="color:#FF6B35; border-left:5px solid #FF6B35;
               padding:6px 0 6px 12px; margin-top:20px; background:#FFF8F5; border-radius:0 8px 8px 0;">
      📍 Bước 1 — Chọn Địa Điểm Lấy Hàng &amp; Giao Hàng
    </h3>"""))

    dd_pickup = widgets.Dropdown(
        options=loc_names, value=loc_names[0],
        description='🏪 Lấy hàng:',
        style={'description_width': '120px'},
        layout=widgets.Layout(width='480px'),
    )
    dd_delivery = widgets.Dropdown(
        options=loc_names, value=loc_names[6],
        description='🏠 Giao hàng:',
        style={'description_width': '120px'},
        layout=widgets.Layout(width='480px'),
    )
    btn_map = widgets.Button(
        description='🗺️  Xem Bản Đồ',
        button_style='info',
        layout=widgets.Layout(width='180px', height='38px'),
    )
    out_map = widgets.Output()

    display(widgets.VBox([
        widgets.HBox([dd_pickup, dd_delivery]),
        btn_map,
        out_map,
    ]))

    # ═══════════════════════════════════════════════════════════
    # PHẦN 2 – ĐIỀU KIỆN
    # ═══════════════════════════════════════════════════════════
    display(HTML("""
    <h3 style="color:#9B59B6; border-left:5px solid #9B59B6;
               padding:6px 0 6px 12px; margin-top:24px; background:#FAF5FF; border-radius:0 8px 8px 0;">
      ⚙️  Bước 2 — Điều Kiện Giao Hàng
    </h3>"""))

    def _slider(desc, val, mn, mx, color='#555'):
        s = widgets.IntSlider(
            value=val, min=mn, max=mx, step=1,
            description=desc,
            style={'description_width': '170px', 'handle_color': color},
            layout=widgets.Layout(width='520px'),
        )
        lbl = widgets.Label(layout=widgets.Layout(min_width='260px'))
        return s, lbl

    s_traffic, lbl_traffic = _slider('🚗 Giao thông (0–10):', 3, 0, 10, '#E74C3C')
    s_weather, lbl_weather = _slider('🌤️ Thời tiết (0–10):', 2, 0, 10, '#3498DB')
    s_prep,    lbl_prep    = _slider('⏱️ Chuẩn bị đơn (phút):', 8, 1, 30, '#F39C12')
    s_fatigue, lbl_fatigue = _slider('😴 Mệt mỏi tài xế (0–10):', 3, 0, 10, '#9B59B6')

    def _upd_traffic(c):
        v = c['new']
        lbl_traffic.value = (
            f'Thấp – Đường thông thoáng ({v}/10)' if v <= 3 else
            f'Trung bình – Tắc nghẽn nhẹ ({v}/10)' if v <= 6 else
            f'Cao – Tắc nghẽn nặng ({v}/10)'
        )
    def _upd_weather(c):
        v = c['new']
        lbl_weather.value = (
            f'Quang đãng ☀️ ({v}/10)' if v <= 3 else
            f'Mưa 🌧️ ({v}/10)'         if v <= 6 else
            f'Giông bão ⛈️ ({v}/10)'
        )
    def _upd_prep(c):
        v = c['new']
        lbl_prep.value = (
            f'Nhanh – dưới 5 phút ({v} phút)' if v <= 5 else
            f'Trung bình ({v} phút)'            if v <= 15 else
            f'Chậm – trên 15 phút ({v} phút)'
        )
    def _upd_fatigue(c):
        v = c['new']
        lbl_fatigue.value = (
            f'Thấp – Tài xế khỏe mạnh ({v}/10)' if v <= 3 else
            f'Trung bình – Hơi mệt ({v}/10)'     if v <= 6 else
            f'Cao – Mệt mỏi nhiều ({v}/10)'
        )

    s_traffic.observe(_upd_traffic, names='value');  _upd_traffic({'new': s_traffic.value})
    s_weather.observe(_upd_weather, names='value');  _upd_weather({'new': s_weather.value})
    s_prep.observe(_upd_prep, names='value');        _upd_prep({'new': s_prep.value})
    s_fatigue.observe(_upd_fatigue, names='value');  _upd_fatigue({'new': s_fatigue.value})

    display(widgets.VBox([
        widgets.HBox([s_traffic, lbl_traffic]),
        widgets.HBox([s_weather, lbl_weather]),
        widgets.HBox([s_prep,    lbl_prep]),
        widgets.HBox([s_fatigue, lbl_fatigue]),
    ]))

    # ═══════════════════════════════════════════════════════════
    # PHẦN 3 – NÚT TÍNH TOÁN
    # ═══════════════════════════════════════════════════════════
    btn_calc = widgets.Button(
        description='🔢  Tính Toán & Đánh Giá',
        button_style='warning',
        layout=widgets.Layout(width='280px', height='52px'),
    )
    btn_reset = widgets.Button(
        description='🔄  Đặt Lại',
        button_style='',
        layout=widgets.Layout(width='130px', height='52px'),
    )
    out_result = widgets.Output()

    display(HTML("<div style='margin-top:22px;'></div>"))
    display(widgets.HBox([btn_calc, btn_reset]))
    display(out_result)

    # ══════════════════════════════════════════════════════════
    # SỰ KIỆN: XEM BẢN ĐỒ
    # ══════════════════════════════════════════════════════════
    def on_show_map(_):
        with out_map:
            clear_output()
            p_name = dd_pickup.value
            d_name = dd_delivery.value
            if p_name == d_name:
                display(HTML(
                    "<p style='color:red;'>⚠️ Vui lòng chọn hai địa điểm khác nhau!</p>"))
                return
            pc = LOCATIONS[p_name]
            dc = LOCATIONS[d_name]
            dist_km = geodesic(pc, dc).km
            display(HTML(
                f"<div style='padding:8px 12px; background:#EBF5FB; border-radius:8px; "
                f"margin:6px 0; font-size:14px;'>"
                f"<b>📏 Khoảng cách thẳng:</b> {dist_km:.2f} km &nbsp;|&nbsp; "
                f"<b>📍</b> {p_name.split('–')[-1].strip()} → "
                f"{d_name.split('–')[-1].strip()}</div>"
            ))
            m = tao_ban_do(pc, dc, p_name, d_name)
            display(m)

    btn_map.on_click(on_show_map)

    # ══════════════════════════════════════════════════════════
    # SỰ KIỆN: TÍNH TOÁN
    # ══════════════════════════════════════════════════════════
    def on_calculate(_):
        with out_result:
            clear_output()
            p_name = dd_pickup.value
            d_name = dd_delivery.value
            if p_name == d_name:
                display(HTML(
                    "<p style='color:red;'>⚠️ Vui lòng chọn hai địa điểm khác nhau!</p>"))
                return

            pc = LOCATIONS[p_name]
            dc = LOCATIONS[d_name]
            dist_km = geodesic(pc, dc).km

            t_val = s_traffic.value
            w_val = s_weather.value
            pr_val = s_prep.value
            f_val = s_fatigue.value

            # ── Suy diễn mờ ──
            del_time, bonus_pct, rating_val = chay_he_thong_mo(
                sim_time, sim_bonus, sim_rating,
                t_val, dist_km, w_val, pr_val, f_val
            )

            # ── Giá tiền ──
            gia_final, weather_lbl, surge = tinh_gia(dist_km, w_val)
            gia_co_ban_km = GIA_CO_BAN + dist_km * GIA_THEO_KM
            tien_thuong   = gia_final * (bonus_pct / 100)
            tong_tien     = gia_final + tien_thuong

            # ── Hiệu suất ──
            stars = hien_thi_sao(rating_val)
            if rating_val >= 4:
                perf_lbl, perf_color, perf_icon = 'Xuất Sắc',  '#27AE60', '🏆'
                perf_desc = '(4–5 ⭐, dịch vụ vượt trội)'
            elif rating_val >= 2.5:
                perf_lbl, perf_color, perf_icon = 'Trung Bình', '#F39C12', '👍'
                perf_desc = '(3 ⭐, hiệu suất chấp nhận được)'
            else:
                perf_lbl, perf_color, perf_icon = 'Kém',        '#E74C3C', '⚠️'
                perf_desc = '(1–2 ⭐, cần cải thiện)'

            # ── Nhãn thời gian ──
            if del_time <= 12:
                time_badge = "<span style='color:#27AE60; font-weight:bold;'>Giao Nhanh ✅</span>"
            elif del_time <= 25:
                time_badge = "<span style='color:#F39C12; font-weight:bold;'>Bình Thường 🕐</span>"
            else:
                time_badge = "<span style='color:#E74C3C; font-weight:bold;'>Giao Chậm ⏳</span>"

            # ── Render kết quả ──
            display(HTML(f"""
<div style="font-family:Arial,sans-serif; max-width:820px; margin:10px 0;">

  <!-- Tiêu đề kết quả -->
  <div style="background:linear-gradient(135deg,#2C3E50,#3498DB);
              color:white; padding:14px 24px; border-radius:14px 14px 0 0; text-align:center;">
    <h2 style="margin:0; font-size:22px;">📊 Kết Quả Đánh Giá Đơn Hàng</h2>
  </div>

  <!-- Lưới 2×2 -->
  <div style="display:grid; grid-template-columns:1fr 1fr; gap:0; border:1px solid #ddd;
              border-top:none; border-radius:0 0 14px 14px; overflow:hidden;">

    <!-- Ô 1: Tuyến đường -->
    <div style="background:#EBF5FB; padding:18px 20px; border-right:1px solid #ddd;
                border-bottom:1px solid #ddd;">
      <h3 style="color:#2980B9; margin:0 0 12px; font-size:16px;">📍 Tuyến Đường</h3>
      <p style="margin:4px 0;"><b>🏪 Lấy hàng:</b><br>
         <span style="color:#555;">{p_name}</span></p>
      <p style="margin:8px 0 4px;"><b>🏠 Giao hàng:</b><br>
         <span style="color:#555;">{d_name}</span></p>
      <p style="margin:10px 0 0; font-size:22px; color:#2980B9; font-weight:bold;">
         📏 {dist_km:.2f} km
      </p>
    </div>

    <!-- Ô 2: Thời gian giao hàng -->
    <div style="background:#F5EEF8; padding:18px 20px; border-bottom:1px solid #ddd;">
      <h3 style="color:#8E44AD; margin:0 0 12px; font-size:16px;">⏰ Thời Gian Giao Hàng Ước Tính</h3>
      <p style="font-size:48px; text-align:center; margin:4px 0; color:#8E44AD; font-weight:bold;
                line-height:1.1;">{del_time:.0f}<span style="font-size:20px;"> phút</span></p>
      <p style="text-align:center; margin:6px 0;">{time_badge}</p>
      <p style="text-align:center; color:#888; font-size:12px; margin:4px 0;">
        (Ngắn: 0–10 | Trung bình: 10–25 | Dài: &gt;25 phút)
      </p>
    </div>

    <!-- Ô 3: Giá tiền -->
    <div style="background:#EAFAF1; padding:18px 20px; border-right:1px solid #ddd;">
      <h3 style="color:#1E8449; margin:0 0 12px; font-size:16px;">💰 Tính Tiền</h3>
      <table style="width:100%; font-size:14px; border-collapse:collapse;">
        <tr><td>Phí cơ bản (15k + {GIA_THEO_KM//1000}k×km)</td>
            <td style="text-align:right;"><b>{gia_co_ban_km:,.0f} ₫</b></td></tr>
        <tr><td>Hệ số thời tiết ({weather_lbl})</td>
            <td style="text-align:right;"><b>× {surge}</b></td></tr>
        <tr><td>Giá sau điều chỉnh</td>
            <td style="text-align:right;"><b>{gia_final:,.0f} ₫</b></td></tr>
        <tr><td>Tiền thưởng ({bonus_pct:.1f}%)</td>
            <td style="text-align:right; color:#27AE60;"><b>+{tien_thuong:,.0f} ₫</b></td></tr>
        <tr style="border-top:2px solid #27AE60;">
          <td style="padding-top:6px; font-weight:bold; font-size:16px;">💳 TỔNG CỘNG</td>
          <td style="text-align:right; font-size:20px; color:#1E8449; font-weight:bold;
                     padding-top:6px;">{tong_tien:,.0f} ₫</td>
        </tr>
      </table>
    </div>

    <!-- Ô 4: Xếp hạng tài xế -->
    <div style="background:#FEF9E7; padding:18px 20px;">
      <h3 style="color:{perf_color}; margin:0 0 12px; font-size:16px;">
        {perf_icon} Xếp Hạng Hiệu Suất Tài Xế
      </h3>
      <p style="font-size:32px; text-align:center; margin:4px 0;">{stars}</p>
      <p style="text-align:center; font-size:28px; font-weight:bold; color:{perf_color};
                margin:4px 0;">{rating_val:.1f} / 5 ⭐</p>
      <p style="text-align:center; font-size:18px; font-weight:bold; color:{perf_color};
                margin:4px 0;">{perf_lbl}</p>
      <p style="text-align:center; color:#888; font-size:12px;">{perf_desc}</p>
    </div>

  </div><!-- /grid -->

  <!-- Tóm tắt đầu vào -->
  <div style="background:#F8F9FA; border:1px solid #dee2e6; border-radius:10px;
              padding:14px 18px; margin-top:14px;">
    <b style="color:#555;">📋 Đầu Vào Logic Mờ:</b>
    <div style="display:flex; flex-wrap:wrap; gap:14px; margin-top:8px; font-size:13px;">
      <span>🚗 Giao thông: <b>{t_val}/10</b></span>
      <span>🌤️ Thời tiết: <b>{w_val}/10</b> ({weather_lbl})</span>
      <span>⏱️ Chuẩn bị đơn: <b>{pr_val} phút</b></span>
      <span>😴 Mệt mỏi: <b>{f_val}/10</b></span>
      <span>🎁 Thưởng: <b>{bonus_pct:.1f}%</b></span>
    </div>
  </div>

</div>
"""))

            # ── Bản đồ kết quả ──
            display(HTML("""
            <h3 style="color:#3498DB; margin-top:20px; border-left:5px solid #3498DB;
                       padding-left:10px;">🗺️ Bản Đồ Tuyến Đường</h3>"""))
            m = tao_ban_do(pc, dc, p_name, d_name)
            display(m)

    def on_reset(_):
        with out_result:
            clear_output()
        with out_map:
            clear_output()
        s_traffic.value = 3
        s_weather.value = 2
        s_prep.value    = 8
        s_fatigue.value = 3
        dd_pickup.value  = loc_names[0]
        dd_delivery.value = loc_names[6]

    btn_calc.on_click(on_calculate)
    btn_reset.on_click(on_reset)

    display(HTML("""
    <div style="margin-top:30px; padding:10px 16px; background:#F0F0F0;
                border-radius:8px; font-size:12px; color:#888; text-align:center;">
      📚 Giáo trình: Trí Tuệ Nhân Tạo Trong Quản Trị Công Nghệ &nbsp;|&nbsp;
      🛠️ Python + scikit-fuzzy + folium + ipywidgets &nbsp;|&nbsp;
      🖥️ Google Colab
    </div>
    """))


# ================================================================
# KHỞI CHẠY ỨNG DỤNG
# ================================================================
tao_ung_dung()


⏳ Đang cài đặt thư viện cần thiết...
✅ Cài đặt hoàn tất!

⚙️  Đang khởi tạo hệ thống Logic Mờ...
✅  Hệ thống sẵn sàng!



Output()